# 🎥 IncuBrix Video B-Roll Engine — Free CogVideoX-2B Colab Server

This notebook runs **THUDM/CogVideoX-2B** on Google Colab's **free Nvidia T4 GPU** and creates a public API tunnel (using Cloudflare Tunnel or ngrok).

You can plug the resulting public URL directly into your local **IncuBrix Video B-Roll Engine** as `AI_VIDEO_GENERATOR_URL` to generate custom B-roll clips for free!

### ⚡ Step 0: Ensure GPU is Enabled
In the menu above, go to **Runtime → Change runtime type → T4 GPU**, then click Save.

In [ ]:
# 1. Verify GPU is available
!nvidia-smi

In [ ]:
# 2. Install required dependencies
!pip install -q --upgrade pip
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
!pip install -q diffusers transformers accelerate sentencepiece protobuf
!pip install -q fastapi uvicorn python-multipart pydantic httpx pyngrok
!pip install -q imageio imageio-ffmpeg

In [ ]:
# 3. Write Server Application Code
server_code = """
import os
import gc
import tempfile
from pathlib import Path
from typing import Optional
import torch
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field
from diffusers import CogVideoXPipeline
from diffusers.utils import export_to_video

app = FastAPI(title="IncuBrix CogVideoX-2B API")
app.add_middleware(CORSMiddleware, allow_origins=["*"], allow_credentials=True, allow_methods=["*"], allow_headers=["*"])

pipeline = None
OUTPUT_DIR = Path("./generated_videos")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

class GenerateRequest(BaseModel):
    prompt: str
    aspect_ratio: Optional[str] = "16:9"
    duration_seconds: Optional[float] = 4.0
    num_inference_steps: Optional[int] = 30
    guidance_scale: Optional[float] = 6.0
    num_frames: Optional[int] = 49

def get_pipeline():
    global pipeline
    if pipeline is not None:
        return pipeline
    print("Loading THUDM/CogVideoX-2b in bfloat16...")
    dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    pipe = CogVideoXPipeline.from_pretrained("THUDM/CogVideoX-2b", torch_dtype=dtype)
    pipe.enable_model_cpu_offload()
    pipe.vae.enable_tiling()
    pipe.vae.enable_slicing()
    pipeline = pipe
    print("Model successfully loaded!")
    return pipeline

@app.on_event("startup")
def startup():
    if torch.cuda.is_available():
        get_pipeline()

@app.get("/health")
def health():
    vram_gb = round(torch.cuda.mem_get_info()[0] / (1024**3), 2) if torch.cuda.is_available() else 0
    return {"status": "ready", "model": "THUDM/CogVideoX-2b", "free_vram_gb": vram_gb}

@app.post("/generate")
def generate(req: GenerateRequest):
    pipe = get_pipeline()
    print(f"Generating video for: {req.prompt}")
    try:
        with torch.inference_mode():
            frames = pipe(
                prompt=req.prompt,
                num_videos_per_prompt=1,
                num_inference_steps=req.num_inference_steps or 30,
                num_frames=req.num_frames or 49,
                guidance_scale=req.guidance_scale or 6.0
            ).frames[0]
        tmp = tempfile.NamedTemporaryFile(suffix=".mp4", delete=False, dir=OUTPUT_DIR)
        export_to_video(frames, tmp.name, fps=8)
        tmp.close()
        gc.collect()
        torch.cuda.empty_cache()
        return FileResponse(tmp.name, media_type="video/mp4", filename="generated.mp4")
    except Exception as e:
        gc.collect()
        torch.cuda.empty_cache()
        raise HTTPException(status_code=500, detail=str(e))
"""
with open("server.py", "w") as f:
    f.write(server_code)
print("[+] server.py generated successfully!")

In [ ]:
# 4. Launch FastAPI in background
import subprocess
import time
import requests

proc = subprocess.Popen(["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8003"])

# Wait for server to come up
for _ in range(60):
    try:
        r = requests.get("http://127.0.0.1:8003/health", timeout=1)
        if r.status_code == 200:
            print("[+] Local FastAPI server is online on port 8003!")
            break
    except Exception:
        time.sleep(2)

In [ ]:
# 5. Expose with Cloudflare Tunnel (100% Free, No Account Required)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

import subprocess
import re

cf_proc = subprocess.Popen(
    ["./cloudflared-linux-amd64", "tunnel", "--url", "http://localhost:8003"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("[*] Starting Cloudflare Tunnel...")
public_url = None
for line in cf_proc.stdout:
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("\n=======================================================")
print(f" ✅ YOUR INCUBRIX AI GENERATOR URL IS: {public_url}")
print("=======================================================")
print("\nCopy this URL and set it in your IncuBrix .env:")
print(f"AI_VIDEO_GENERATOR_URL={public_url}")